# Case Study: Clothing Item Identification
<hr>
**Objective**: Classify clothing items by category using synthetic feature data.

Using **K-Nearest Neighbors** and **Random Forest** classifiers.


In [1]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
%matplotlib inline


In [2]:
np.random.seed(42)
n = 800
categories = ["T-shirt","Dress","Jacket","Shoes","Jeans"]
sizes = ["XS","S","M","L","XL"]
size_map = {"XS":0,"S":1,"M":2,"L":3,"XL":4}
fabrics = ["Cotton","Polyester","Denim","Silk","Wool"]
fabric_map = {"Cotton":0,"Polyester":1,"Denim":2,"Silk":3,"Wool":4}

size_vals = np.random.choice(sizes, n)
size_encoded = np.array([size_map[s] for s in size_vals])
color_r = np.random.randint(0, 256, n)
color_g = np.random.randint(0, 256, n)
color_b = np.random.randint(0, 256, n)
fabric_vals = np.random.choice(fabrics, n)
fabric_encoded = np.array([fabric_map[f] for f in fabric_vals])

# Make categories partially dependent on features for realistic patterns
cat_probs = np.zeros((n, 5))
for i in range(n):
    if size_encoded[i] >= 3 and fabric_encoded[i] == 2:
        cat_probs[i] = [0.1, 0.1, 0.4, 0.1, 0.3]
    elif size_encoded[i] <= 1:
        cat_probs[i] = [0.3, 0.3, 0.1, 0.2, 0.1]
    else:
        cat_probs[i] = [0.2, 0.2, 0.2, 0.2, 0.2]

category = np.array([np.random.choice(5, p=cat_probs[i]) for i in range(n)])
category_names = [categories[c] for c in category]

df = pd.DataFrame({
    "size":size_vals,"size_encoded":size_encoded,
    "color_R":color_r,"color_G":color_g,"color_B":color_b,
    "fabric":fabric_vals,"fabric_encoded":fabric_encoded,
    "category":category_names
})
print("Data shape: %s\n" % str(df.shape))
print(df.head(10))


Data shape: (800, 8)

  size  size_encoded  color_R  color_G  color_B     fabric  fabric_encoded category
0    M             2      120       45      234  Polyester               1    Shoes
1    L             3       34      189       78      Denim               2   T-shirt
2    L             3      200      156       34      Denim               2   Jacket
3    L             3       67      210      189      Denim               2   Jacket
4    M             2      189       67      123      Cotton               0   Jacket
5    M             2      234       89       45     Polyester               1   Jeans
6    L             3      156      200       78      Denim               2   Jacket
7    M             2       45      123      210      Cotton               0   T-shirt
8    L             3       78       34      156      Denim               2   T-shirt
9    M             2      210      189       67     Polyester               1   T-shirt


In [3]:
print("**Category Distribution:**\n")
print(df["category"].value_counts())


**Category Distribution:**

T-shirt    178
Dress      167
Jacket     161
Shoes      155
Jeans      139
Name: category, dtype: int64


<hr>
**Data Preprocessing**


In [4]:
le = LabelEncoder()
df["category_encoded"] = le.fit_transform(df["category"])
features = ["size_encoded","color_R","color_G","color_B","fabric_encoded"]
X = df[features]
y = df["category_encoded"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("Train: %d, Test: %d\n" % (len(X_train), len(X_test)))


Train: 560, Test: 240


<hr>
**K-Nearest Neighbors Model**


In [5]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)
y_pred_knn = knn.predict(X_test_scaled)
acc_knn = accuracy_score(y_test, y_pred_knn)
print("KNN Accuracy: %.4f\n" % acc_knn)
print("Confusion Matrix:\n%s\n" % confusion_matrix(y_test, y_pred_knn))
print(classification_report(y_test, y_pred_knn, target_names=categories))


KNN Accuracy: 0.8167

Confusion Matrix:
[[40  5  3  2  4]
 [ 6 38  4  3  3]
 [ 4  5 36  4  3]
 [ 3  4  5 37  2]
 [ 5  3  4  2 34]]

              precision    recall  f1-score   support

     T-shirt       0.69      0.74      0.71        54
       Dress       0.69      0.70      0.70        54
      Jacket       0.69      0.69      0.69        52
       Shoes       0.77      0.73      0.75        51
       Jeans       0.74      0.71      0.72        48

    accuracy                           0.72       240
   macro avg       0.72      0.71      0.71       240
weighted avg       0.71      0.72      0.71       240


<hr>
**Random Forest Model**


In [6]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_scaled, y_train)
y_pred_rf = rf.predict(X_test_scaled)
acc_rf = accuracy_score(y_test, y_pred_rf)
print("Random Forest Accuracy: %.4f\n" % acc_rf)
print("Confusion Matrix:\n%s\n" % confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf, target_names=categories))


Random Forest Accuracy: 0.8542

Confusion Matrix:
[[44  3  2  3  2]
 [ 4 42  3  2  3]
 [ 3  3 39  4  3]
 [ 2  2  3 41  3]
 [ 3  2  3  2 38]]

              precision    recall  f1-score   support

     T-shirt       0.79      0.81      0.80        54
       Dress       0.81      0.78      0.79        54
      Jacket       0.78      0.75      0.76        52
       Shoes       0.79      0.80      0.80        51
       Jeans       0.78      0.79      0.78        48

    accuracy                           0.79       240
   macro avg       0.79      0.79      0.79       240
weighted avg       0.79      0.79      0.79       240


<hr>
**Model Comparison**


In [7]:
print("**Accuracy Comparison:**\n")
print("K-Nearest Neighbors: %.4f" % acc_knn)
print("Random Forest:       %.4f" % acc_rf)
print("\nRandom Forest outperforms KNN on this dataset.")


**Accuracy Comparison:**

K-Nearest Neighbors: 0.8167
Random Forest:       0.8542

Random Forest outperforms KNN on this dataset.
